CIFAR = Canadian Institute For Advanced Research

In ML, when people say CIFAR, they usually mean the image datasets created by them.

The most common one is:

CIFAR-10

A small image dataset used to train/test image classifiers.

Contains:

60,000 images
Size: 32×32 pixels (very small)
10 classes

Classes are:

Plane

Car

Bird

Cat

Deer

Dog

Frog

Horse

Ship

Truck



ImageNet

ImageNet = dataset

Huge collection of ~1.2 million labeled images across 1000 classes (dog, cat, car, etc.)

Used to train powerful vision models.


Think:
ImageNet = the textbook + practice questions



ResNet

ResNet = neural network architecture/model

A CNN design made to classify images effectively, especially deep networks.

Can be trained on ImageNet.

Think:
ResNet = the student/brain that studies the textbook

Transfer learning
Use pretrained model knowledge instead of learning from zero.
Why it works
Image features like:


edges


textures


shapes


are universal.
ResNet structure
Edges → textures → shapes → classifier
What you replace
Only:
FC layer
Phase 1
Freeze backbone, train head.
Phase 2
Unfreeze and gently fine-tune.

One-line memory hook

Transfer learning = hiring an experienced visual expert and teaching only your specific task.


In [11]:
# import torch
# import torch.nn as nn
# import torchvision
# import torchvision.models as models

# # Load ResNet18 pretrained on ImageNet
# model = models.resnet18(weights='IMAGENET1K_V1')

# # Inspect the full architecture
# print(model)

# # How many parameters total?
# total = sum(p.numel() for p in model.parameters())
# print(f"\nTotal params: {total:,}")   # ~11 million

In [12]:
#  # The last layer — outputs 1000 ImageNet class scores
# print(model.fc)
# # Linear(in_features=512, out_features=1000, bias=True)

# # Check input to the fc layer
# print("FC input features:", model.fc.in_features)   # 512

In [13]:
# # Swap the 1000-class head for a 10-class head
# model.fc = nn.Sequential(
#     nn.Linear(512, 256),
#     nn.ReLU(),
#     nn.Dropout(0.3),
#     nn.Linear(256, 10)
# )

# # Confirm the change
# print(model.fc)
# print("New total params:", sum(p.numel() for p in model.parameters()))

In [14]:
# # Freeze ALL layers first
# for param in model.parameters():
#     param.requires_grad = False

# # Unfreeze only the new head
# for param in model.fc.parameters():
#     param.requires_grad = True

# # Check: how many params are trainable?
# trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
# frozen    = sum(p.numel() for p in model.parameters() if not p.requires_grad)
# print(f"Trainable: {trainable:,}  |  Frozen: {frozen:,}")

This section is basically:

Take pretrained ResNet → modify it for CIFAR-10 → freeze most of it → train only the new part.

Load pretrained model
→ borrow ImageNet knowledge

Replace FC layer
→ adapt to CIFAR-10 classes

Freeze backbone
→ protect pretrained features

Train head only
→ fast + stable transfer learning phase 1.

In [15]:
# import torchvision.transforms as transforms

# train_transform = transforms.Compose([
#     transforms.Resize(64),                        # upsample small images
#     transforms.RandomHorizontalFlip(),
#     transforms.RandomCrop(64, padding=8),
#     transforms.ToTensor(),
#     transforms.Normalize((0.485,0.456,0.406),     # ImageNet stats — use these
#                          (0.229,0.224,0.225))
# ])
# val_transform = transforms.Compose([
#     transforms.Resize(64),
#     transforms.ToTensor(),
#     transforms.Normalize((0.485,0.456,0.406),(0.229,0.224,0.225))
# ])

# train_dataset = torchvision.datasets.CIFAR10('./data', train=True,  transform=train_transform)
# val_dataset   = torchvision.datasets.CIFAR10('./data', train=False, transform=val_transform)
# train_loader  = torch.utils.data.DataLoader(train_dataset, batch_size=64, shuffle=True,  num_workers=0)
# val_loader    = torch.utils.data.DataLoader(val_dataset,   batch_size=64, shuffle=False, num_workers=0)

Big Picture

Remember:

ResNet was trained on:

ImageNet images

CIFAR is different:

32×32 tiny images

Problem:

ResNet expects data in a certain format.

So before training:

We must make CIFAR look more like ImageNet.

That is what these transforms do.

----------
What are transforms?

Think:

Raw image
    ↓
Transform pipeline
    ↓
Model-ready image

Transforms = preprocessing steps for images.

Just like tabular ML had:

scaling
imputation
encoding

Computer vision has:

resize
augment
normalize

Same idea.


----------------
Overall Flow
CIFAR image
      ↓
Resize
      ↓
Flip + Crop (augmentation)
      ↓
Tensor
      ↓
ImageNet normalization
      ↓
Batch with DataLoader
      ↓
Ready for ResNet

Quick Summary

Resize

→ make CIFAR size suitable for ResNet

Flip + Crop

→ data augmentation, reduce overfitting

ToTensor

→ convert image to tensor

Normalize

→ scale pixels using ImageNet stats because pretrained ResNet expects them

DataLoader

→ batch + shuffle + efficient loading.

So this whole section is simply:

Make CIFAR images look like the kind of images ResNet was trained to understand.

In [16]:
# import torch.optim as optim

# device    = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# model     = model.to(device)
# criterion = nn.CrossEntropyLoss()

# # Only pass trainable params to optimiser
# optimizer = optim.Adam(
#     filter(lambda p: p.requires_grad, model.parameters()),
#     lr=0.001
# )

# def train_epoch(model, loader, optimizer, criterion, device):
#     model.train()
#     correct, total, total_loss = 0, 0, 0.0
#     for imgs, labels in loader:
#         imgs, labels = imgs.to(device), labels.to(device)
#         optimizer.zero_grad()
#         out  = model(imgs)
#         loss = criterion(out, labels)
#         loss.backward()
#         optimizer.step()
#         total_loss += loss.item()
#         _, preds = torch.max(out, 1)
#         correct  += (preds == labels).sum().item()
#         total    += labels.size(0)
#     return total_loss/len(loader), correct/total

# def eval_epoch(model, loader, criterion, device):
#     model.eval()
#     correct, total, total_loss = 0, 0, 0.0
#     with torch.no_grad():
#         for imgs, labels in loader:
#             imgs, labels = imgs.to(device), labels.to(device)
#             out  = model(imgs)
#             loss = criterion(out, labels)
#             total_loss += loss.item()
#             _, preds = torch.max(out, 1)
#             correct  += (preds == labels).sum().item()
#             total    += labels.size(0)
#     return total_loss/len(loader), correct/total

# print("=== Phase 1: Head only (frozen backbone) ===")
# for epoch in range(5):
#     tl, ta = train_epoch(model, train_loader, optimizer, criterion, device)
#     vl, va = eval_epoch(model, val_loader, criterion, device)
#     print(f"Epoch {epoch+1} | train: {ta:.3f} | val: {va:.3f}")

Same story again:

Clear old gradients->
Predict->
Compute loss->
Backpropagation->
Update weights

Only the head weights change.

Phase 1 = frozen backbone + train head only

ResNet already knows visual features

Train only new classifier layer

CrossEntropyLoss measures classification error

Adam updates only trainable params

train_epoch() = learn

eval_epoch() = test

5 epochs teach the head to map pretrained features → CIFAR-10 labels

Core idea:

First teach the new head before modifying the pretrained brain.

In [17]:
# # Unfreeze ALL layers
# for param in model.parameters():
#     param.requires_grad = True

# # Use a much lower learning rate — the pretrained weights are good,
# # you just want to nudge them slightly, not overwrite them
# optimizer = optim.Adam([
#     {'params': model.fc.parameters(),  'lr': 1e-3},   # head: normal lr
#     {'params': [p for name, p in model.named_parameters()
#                 if 'fc' not in name],  'lr': 1e-4},   # backbone: 10x lower lr
# ])
# scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=10)

# best_val_acc = 0
# print("\n=== Phase 2: Full fine-tuning ===")
# for epoch in range(10):
#     tl, ta = train_epoch(model, train_loader, optimizer, criterion, device)
#     vl, va = eval_epoch(model, val_loader, criterion, device)
#     scheduler.step()
#     print(f"Epoch {epoch+1:2d} | train: {ta:.3f} | val: {va:.3f}")

#     if va > best_val_acc:
#         best_val_acc = va
#         torch.save(model.state_dict(), 'resnet18_cifar10.pth')

# print(f"\nBest val accuracy: {best_val_acc:.3f}")

Phase 2 = full fine-tuning

Flow:

Phase 1:
Train head only

        ↓

Phase 2:
Unfreeze all layers

        ↓

Small LR for backbone
Large LR for head

        ↓

Fine-tune pretrained knowledge

        ↓

Save best-performing model

Core idea:

Don't rebuild vision from scratch — gently adapt pretrained intelligence to your task.

In [18]:
# print("=" * 45)
# print("MODEL COMPARISON — CIFAR-10")
# print("=" * 45)
# print(f"{'Model':<28} {'Val Acc':>8}  {'Epochs':>7}")
# print("-" * 45)
# print(f"{'Day 3 — Basic CNN':<28} {'~65-70%':>8}  {'10':>7}")
# print(f"{'Day 4 — CNN + BN + Dropout':<28} {'~73-78%':>8}  {'15-20':>7}")
# print(f"{'Day 5 — ResNet18 (transfer)':<28} {f'{best_val_acc*100:.1f}%':>8}  {'15':>7}")
# print("=" * 45)

Quick Summary

This code:

✅ Creates a clean comparison table

✅ Uses f-string alignment (<, >)

✅ Shows your final CIFAR-10 results

✅ Helps compare Basic CNN vs Improved CNN vs Transfer Learning (ResNet)

Main takeaway:

Transfer learning usually wins — higher accuracy with less training effort.